# LSTM vs GRU vs Vanilla RNN — A Controlled Ablation

**SENTINEL-CXR** — Uncertainty-Aware Chest Radiograph Triage
Deep Learning (MAIB AI 114) · Prof Anshul Gupta · S P Jain School of Global Management, Dubai

| Group member | Student ID |
|---|---|
| Krishna Mathur | AS25DXB018 |
| Atharva Soundankar | AS25DXB020 |
| Yash Petkar | AS25DXB021 |

---

**Syllabus mapping — Week 5: Long Short-Term Memory Networks**

Three recurrent cells, one dataset, identical training conditions. This directly
addresses learning outcome B, *evaluate the performance of diverse deep learning
models*.

The comparison is only meaningful if everything except the cell is held fixed:
same splits, same seed, same hidden size, same optimiser, same epochs. The
notebook also measures gradient flow, which is the mechanism that separates the
three — a vanilla RNN's gradient decays multiplicatively through time, while the
LSTM's additive cell state preserves it.


In [ ]:
# ── Environment ───────────────────────────────────────────────────────
# Runs on Colab free tier (T4). Nothing here needs a paid runtime.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "torchxrayvision", "scikit-learn", "seaborn"],
        check=False,
    )

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 20260812
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
})
INSTRUMENT, STAT = "#2E9CB8", "#D64541"

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
# NIH ChestX-ray14: 112,120 frontal radiographs, 30,805 patients, 14 labels.
# Kaggle: https://www.kaggle.com/datasets/nih-chest-xrays/data
#
# In Colab, the fastest route is the Kaggle API:
#   from google.colab import files; files.upload()      # kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d nih-chest-xrays/data -p /content/nih --unzip

DATA_DIR = os.environ.get("NIH_DIR", "/content/nih")
META = os.path.join(DATA_DIR, "Data_Entry_2017.csv")

PATHOLOGIES = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Effusion",
               "Emphysema","Fibrosis","Hernia","Infiltration","Mass","Nodule",
               "Pleural_Thickening","Pneumonia","Pneumothorax"]

def load_metadata(path=META):
    """Load the label CSV and expand `Finding Labels` into 14 binary columns."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for p in PATHOLOGIES:
        df[p] = df["Finding Labels"].str.contains(p, regex=False).astype(int)
    df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
    # Ages above ~100 in this dataset are data-entry errors, not centenarians.
    df = df[(df["Patient Age"] > 0) & (df["Patient Age"] < 100)]
    return df

def patient_disjoint_split(df, fracs=(0.70, 0.10, 0.20), seed=SEED):
    """Split by Patient ID — NEVER by image.

    A patient contributes 3-4 follow-up studies. Splitting by image places the
    same patient's scans on both sides of the boundary, so the model can
    memorise the patient rather than the pathology. Every metric then reports a
    number that will not survive contact with a new hospital. This is the most
    common methodological error in published work on ChestX-ray14.
    """
    patients = df["Patient ID"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    a, b = int(fracs[0] * n), int((fracs[0] + fracs[1]) * n)
    sets = (set(patients[:a]), set(patients[a:b]), set(patients[b:]))
    train, cal, test = (df[df["Patient ID"].isin(s)].copy() for s in sets)
    assert not (set(train["Patient ID"]) & set(test["Patient ID"])), "patient leak"
    return train, cal, test

## 1. The comparison harness

In [ ]:
class RecurrentHead(nn.Module):
    """One class, three cells — so nothing but the cell can differ."""

    def __init__(self, cell="lstm", input_dim=1024, hidden=256, layers=2,
                 bidirectional=False, dropout=0.3):
        super().__init__()
        cls = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[cell]
        kw = dict(input_size=input_dim, hidden_size=hidden, num_layers=layers,
                  batch_first=True, bidirectional=bidirectional,
                  dropout=dropout if layers > 1 else 0.0)
        if cell == "rnn": kw["nonlinearity"] = "tanh"
        self.rnn, self.cell = cls(**kw), cell
        out_dim = hidden * (2 if bidirectional else 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(out_dim, 14))

    def forward(self, x, lengths=None):
        out, _ = self.rnn(x)
        if lengths is not None:
            idx = (lengths - 1).clamp(min=0)
            last = out[torch.arange(out.size(0)), idx]
        else:
            last = out[:, -1]
        return self.head(last)

for cell in ["rnn", "gru", "lstm"]:
    m = RecurrentHead(cell=cell)
    n = sum(p.numel() for p in m.parameters())
    print(f"{cell:5s} {n:>10,} parameters")
print("\nLSTM has ~4x the recurrent parameters of a vanilla RNN (4 gates),")
print("GRU ~3x (reset, update, candidate). Parameter count is itself a")
print("confound, so the comparison also reports accuracy per parameter.")

## 2. Gradient flow — the actual mechanism

In [ ]:
def gradient_flow(cell, seq_len=60, hidden=64, trials=12):
    """Gradient magnitude reaching each timestep, averaged over random inits.

    The vanishing-gradient problem made visible: how much signal from the loss
    at the end of the sequence survives back to the beginning.

    Averaged over `trials` seeds because a single initialisation is very noisy
    — the spread across seeds is larger than the gap between GRU and LSTM, so
    any conclusion drawn from one run would not be reproducible.
    """
    runs = []
    for t in range(trials):
        torch.manual_seed(SEED + t)
        model = RecurrentHead(cell=cell, input_dim=16, hidden=hidden,
                              layers=1, dropout=0.0)
        x = torch.randn(8, seq_len, 16, requires_grad=True)
        model(x).sum().backward()
        runs.append(x.grad.abs().mean(dim=(0, 2)).detach().numpy())
    runs = np.stack(runs)
    return runs.mean(0), runs.std(0)

fig, ax = plt.subplots(figsize=(7, 3.2))
rows = []
for cell, colour in zip(["rnn", "gru", "lstm"], ["#8A9299", "#D9903F", INSTRUMENT]):
    g, sd = gradient_flow(cell)
    ax.semilogy(g, label=cell.upper(), color=colour, lw=1.6)
    ax.fill_between(range(len(g)), np.maximum(g - sd, 1e-30), g + sd,
                    color=colour, alpha=0.15)
    # Survival ratio: gradient at the START relative to the END. Smaller means
    # more signal was lost travelling back through time.
    rows.append((cell, g[0], g[-1], g[0] / max(g[-1], 1e-30)))

ax.set_xlabel("timestep"); ax.set_ylabel("mean |gradient| (log)")
ax.set_title("Gradient reaching each timestep, 60 steps, mean of 12 seeds")
ax.legend(); plt.tight_layout(); plt.show()

df = pd.DataFrame(rows, columns=["cell", "grad@t=0", "grad@t=59", "survival ratio"])
print(df.to_string(index=False, float_format=lambda v: f"{v:.3e}"))

best = df.loc[df["survival ratio"].idxmax(), "cell"]
print(f"\nGated cells retain far more gradient at early timesteps than the")
print(f"vanilla RNN, whose product of Jacobians decays geometrically through")
print(f"time. Best survival ratio in this run: {best.upper()}.")
print("\nGRU and LSTM are close here and their ordering is not stable across")
print("seeds, so no claim is made that one dominates the other on this probe.")
print("The architectural argument (Hochreiter & Schmidhuber 1997) is that the")
print("LSTM cell state carries gradient ADDITIVELY rather than multiplicatively;")
print("the downstream AUROC comparison, not this probe, is what decides which")
print("cell to deploy.")

## 3. Results table for the report

In [ ]:
def ablation_table(results):
    """results: {cell: {"auroc": float, "params": int, "epoch_s": float}}"""
    df = pd.DataFrame([
        {"cell": k.upper(), "macro AUROC": v["auroc"], "parameters": v["params"],
         "AUROC per 1M params": v["auroc"] / (v["params"] / 1e6),
         "sec/epoch": v["epoch_s"]}
        for k, v in results.items()
    ])
    print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    return df

# Fill from your training runs. Reporting parameter count and time alongside
# AUROC prevents the ablation from rewarding a model purely for being larger.
print("Populate `results` from the training loop, then call ablation_table().")

---

### References for this notebook

- Hochreiter, S. & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*.
- Greff, K. et al. (2017). LSTM: a search space odyssey. *IEEE TNNLS*.
- Chen, J. et al. (2020). LSTM for traffic speed prediction. *Transportation Research Part C*.

---

*SENTINEL-CXR is a student research prototype. It is not a medical device and
must not be used for clinical decisions.*
